# Emotions Binary Classification Pipeline (Love Detection)

This notebook implements an **active learning pipeline** for binary emotion classification, following the same **LTS (Learning Through Sampling) structure** used in the 20 Newsgroups workflow:

## Pipeline Flow
1. **Configuration**: Set target emotion and paths
2. **Data Preparation**: Prepare binary classification dataset (target=1, others=0)
3. **Module Verification**: Ensure all required pipeline components are available
4. **Active Learning Loop**: Cluster → Sample → Pseudo-label → Fine-tune → Evaluate
5. **Final Evaluation**: Report metrics on held-out test set

## Task Definition
- **Label 1**: Target emotion = `love` (or configurable)
- **Label 0**: All other emotions
- **Objective**: Train a binary classifier using active learning with Thompson sampling

In [ ]:
## Step 1: Environment Setup (Colab)

## Step 1: Environment Setup (Colab)

In [ ]:
import os
import sys

REPO_URL = "https://github.com/ShenyaoZhang/DS-GA-3001-Data-Engineering-Project.git"
BRANCH = "emotions-rec-sentiments"

DRIVE_ROOT = "/content/drive/MyDrive"
REPO_ROOT = os.path.join(DRIVE_ROOT, "DS-GA-3001-Data-Engineering-Project")
EXPERIMENT_ROOT = os.path.join(REPO_ROOT, "emotions_rec")

SRC_DIR = os.path.join(EXPERIMENT_ROOT, "src")
PROMPTS_DIR = os.path.join(EXPERIMENT_ROOT, "prompts")
DATA_PROCESSED_DIR = os.path.join(EXPERIMENT_ROOT, "data", "processed")

if not os.path.exists(REPO_ROOT):
    !git clone -b {BRANCH} {REPO_URL} "{REPO_ROOT}"
else:
    %cd "{REPO_ROOT}"
    !git fetch origin
    !git checkout {BRANCH}
    !git pull

%cd "{EXPERIMENT_ROOT}"
os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("Experiment root:", EXPERIMENT_ROOT)

import subprocess
from datetime import datetime

LOG_DIR = os.path.join(EXPERIMENT_ROOT, "logs")
os.makedirs(LOG_DIR, exist_ok=True)
print("Logs directory (persisted under your Drive repo):", LOG_DIR)


def run_logged(cmd, log_path, cwd=None, notebook_tail_lines=1200):
    """Append full subprocess output to ``log_path``; print only the last lines in the notebook (reduces Colab UI freezes)."""
    log_path = os.path.abspath(log_path)
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    stamp = datetime.now().isoformat(timespec="seconds")
    lines = []
    cmd_list = [str(x) for x in cmd]
    with open(log_path, "a", encoding="utf-8", errors="replace") as lf:
        lf.write(f"\n{'=' * 72}\n[{stamp}] $ {' '.join(cmd_list)}\n{'=' * 72}\n")
        lf.flush()
        try:
            p = subprocess.Popen(
                cmd_list,
                cwd=cwd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                encoding="utf-8",
                errors="replace",
                bufsize=1,
            )
            assert p.stdout is not None
            for line in iter(p.stdout.readline, ""):
                lf.write(line)
                lf.flush()
                lines.append(line)
            p.stdout.close()
            rc = p.wait()
        except Exception as exc:
            lf.write(f"\n[run_logged] exception: {exc!r}\n")
            lf.flush()
            print(f"[run_logged] exception (see log): {exc!r}")
            raise

    if notebook_tail_lines is not None and notebook_tail_lines > 0:
        tail = lines[-notebook_tail_lines:]
        if len(lines) > notebook_tail_lines:
            print(f"\n... ({len(lines) - notebook_tail_lines} earlier lines only in log file)\n")
    else:
        tail = lines

    try:
        sys.stdout.write("".join(tail))
        sys.stdout.flush()
    except (BrokenPipeError, UnicodeEncodeError) as exc:
        print(f"[notebook stdout skipped: {exc!r}]")

    print(f"\n[exit {rc}] full log: {log_path}")
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd_list)
    return rc



## Step 2: Configuration & Paths

Set up repository paths, target emotion, and experiment directories.

In [ ]:
from getpass import getpass
import os

token = getpass("Enter your HF token: ")
os.environ["HF_TOKEN"] = token
os.environ["HUGGINGFACE_HUB_TOKEN"] = token
print("HF token set.")


## Step 3: Authentication & Dependencies

Long installs stream to **`emotions_rec/logs/`** on your Drive (`pip_*.log`, etc.). The notebook prints only the **tail** of each subprocess so huge logs do not freeze Colab; open the `.log` files for full output.


In [ ]:
%cd "{EXPERIMENT_ROOT}"
run_logged(
    ["python", "-m", "pip", "install", "-r", os.path.join(REPO_ROOT, "requirements.txt")],
    os.path.join(LOG_DIR, "pip_requirements.log"),
    cwd=EXPERIMENT_ROOT,
    notebook_tail_lines=500,
)
run_logged(
    ["python", "-m", "pip", "install", "datasets"],
    os.path.join(LOG_DIR, "pip_datasets.log"),
    cwd=EXPERIMENT_ROOT,
    notebook_tail_lines=500,
)


### Install Requirements

In [ ]:
TARGET_LABEL = "love"

%cd "{EXPERIMENT_ROOT}"
run_logged(
    ["python", "-u", "scripts/prepare_emotions_binary.py", "--label", TARGET_LABEL],
    os.path.join(LOG_DIR, "prepare_emotions_binary.log"),
    cwd=EXPERIMENT_ROOT,
    notebook_tail_lines=800,
)

TRAIN_FILE = f"data/processed/emotions_{TARGET_LABEL}_smoke_train"
VAL_FILE = f"data/processed/emotions_{TARGET_LABEL}_smoke_validation.csv"
TEST_FILE = f"data/processed/emotions_{TARGET_LABEL}_test.csv"
# Full validation for threshold tuning / reporting (distribution closer to benchmark test than smoke val)
FULL_VAL_TUNING_PATH = f"data/processed/emotions_{TARGET_LABEL}_validation.csv"

print("Train:", TRAIN_FILE)
print("Val:  ", VAL_FILE)
print("Test: ", TEST_FILE)


## Step 4: Data Preparation (Binary Classification Task)

Configure target emotion and prepare binary dataset (love=1, others=0).

In [ ]:
import pandas as pd

train_df = pd.read_csv(TRAIN_FILE + ".csv")
val_df = pd.read_csv(VAL_FILE)
test_df = pd.read_csv(TEST_FILE)

print("train distribution:", train_df["label"].value_counts().to_dict())
print("val distribution:", val_df["label"].value_counts().to_dict())
print("test distribution:", test_df["label"].value_counts().to_dict())


### Verify Data Splits & Distributions

In [ ]:
EXPECTED = [
    "preprocessing.py",
    "LDA.py",
    "labeling.py",
    "random_sampling.py",
    "thompson_sampling.py",
    "fine_tune.py",
    "main_cluster_emotion_binary.py",
    "eval_emotion_binary.py",
]

missing = []
for name in EXPECTED:
    path = os.path.join(SRC_DIR, name)
    ok = os.path.exists(path)
    print(f"{name}: {'FOUND' if ok else 'MISSING'}")
    if not ok:
        missing.append(name)

script_ok = os.path.exists(os.path.join(EXPERIMENT_ROOT, "scripts", "prepare_emotions_binary.py"))
print("scripts/prepare_emotions_binary.py:", "FOUND" if script_ok else "MISSING")
if not script_ok:
    missing.append("scripts/prepare_emotions_binary.py")

if missing:
    raise FileNotFoundError(f"Missing: {missing}")


## Step 5: Module & Pipeline Verification

Verify all required components (preprocessing, LDA, labeling, sampling, fine-tuning) are present and compile correctly.

In [ ]:
targets = [os.path.join(SRC_DIR, f) for f in EXPECTED]
run_logged(
    ["python", "-m", "py_compile", *targets],
    os.path.join(LOG_DIR, "py_compile.log"),
    cwd=EXPERIMENT_ROOT,
    notebook_tail_lines=None,
)
print("py_compile finished (see log above and file).")


### Compile Python Modules

In [ ]:
# Full training log: emotions_rec/logs/active_learning_train.log (tail printed below to avoid Colab freezing).
os.chdir(EXPERIMENT_ROOT)
train_cmd = [
    "python",
    "-u",
    "src/main_cluster_emotion_binary.py",
    "-sample_size",
    "200",
    "-filename",
    TRAIN_FILE,
    "-val_path",
    VAL_FILE,
    "-balance",
    "False",
    "-sampling",
    "thompson",
    "-filter_label",
    "True",
    "-model_finetune",
    "bert-base-uncased",
    "-labeling",
    "qwen",
    "-model",
    "text",
    "-baseline",
    "0.10",
    "-metric",
    "f1_pos",
    "-cluster_size",
    "8",
    "-positive_label",
    TARGET_LABEL,
    "-hf_model_id",
    "Qwen/Qwen2.5-3B-Instruct",
    "-max_iterations",
    "3",
    "-num_train_epochs",
    "2",
    "-max_length",
    "128",
    "-batch_size",
    "16",
    "-confidence_threshold",
    "0.40",
]
print("val_path will be:", VAL_FILE)
run_logged(
    train_cmd,
    os.path.join(LOG_DIR, "active_learning_train.log"),
    cwd=EXPERIMENT_ROOT,
    notebook_tail_lines=1500,
)


### Optional: random sampling baseline (fair vs Thompson)

Same **`TRAIN_FILE`**, **`VAL_FILE`**, and **`TEST_FILE`** as above (same validation set during training; same held-out test when you eval). Uses **`-run_tag random`** so results/checkpoints/sampler state do not overwrite Thompson.

Recommended order: run Thompson **first** so LDA exists; then set **`RUN_RANDOM_COMPARE = True`** in the next cell and run it.


In [ ]:
RUN_RANDOM_COMPARE = False  # set True to train random sampler baseline after Thompson

if RUN_RANDOM_COMPARE:
    os.chdir(EXPERIMENT_ROOT)
    train_random_cmd = [
        "python",
        "-u",
        "src/main_cluster_emotion_binary.py",
        "-sample_size",
        "200",
        "-filename",
        TRAIN_FILE,
        "-val_path",
        VAL_FILE,
        "-balance",
        "False",
        "-sampling",
        "random",
        "-run_tag",
        "random",
        "-filter_label",
        "True",
        "-model_finetune",
        "bert-base-uncased",
        "-labeling",
        "qwen",
        "-model",
        "text",
        "-baseline",
        "0.10",
        "-metric",
        "f1_pos",
        "-cluster_size",
        "8",
        "-positive_label",
        TARGET_LABEL,
        "-hf_model_id",
        "Qwen/Qwen2.5-3B-Instruct",
        "-max_iterations",
        "3",
        "-num_train_epochs",
        "2",
        "-max_length",
        "128",
        "-batch_size",
        "16",
        "-confidence_threshold",
        "0.40",
    ]
    run_logged(
        train_random_cmd,
        os.path.join(LOG_DIR, "active_learning_train_random.log"),
        cwd=EXPERIMENT_ROOT,
        notebook_tail_lines=1500,
    )
else:
    print("Skipping random baseline (set RUN_RANDOM_COMPARE = True).")


## Step 6: Active Learning Training Pipeline

Run the main LTS loop: **Cluster → Sample → Pseudo-label → Fine-tune → Evaluate**

This implements active learning with Thompson sampling to iteratively improve the binary love emotion classifier.

Full training output is appended to **`logs/active_learning_train.log`** on Drive; the code cell prints only the **last ~1500 lines** here so Colab stays responsive.


## Step 6b: Per-cluster validation metrics

After training, inspect **validation metrics by LDA cluster** (`validation_per_cluster` in the JSON). Each row reflects performance on validation examples assigned to that topic id.

**Requirements:** `{TRAIN_FILE}_lda.pkl` must exist (created when LDA is first built). If you only have `{TRAIN_FILE}_lda.csv`, delete the CSV once and rerun training so the pickle is written.

Results file: `{TRAIN_FILE}_binary_{TARGET_LABEL}_model_results.json`.

Optional random baseline (same splits): `{TRAIN_FILE}_binary_{TARGET_LABEL}_random_model_results.json` — reuse the Step 6b/6c cells by pointing `MODEL_RESULTS_PATH` at that file.

**Step 6c** (below) saves metric plots to `logs/figures/`.


In [ ]:
import json
from pathlib import Path

import pandas as pd

%cd "{EXPERIMENT_ROOT}"

MODEL_RESULTS_PATH = Path(EXPERIMENT_ROOT) / f"{TRAIN_FILE}_binary_{TARGET_LABEL}_model_results.json"
LDA_PKL = Path(EXPERIMENT_ROOT) / f"{TRAIN_FILE}_lda.pkl"

print("Model results:", MODEL_RESULTS_PATH)
print("LDA pickle:", LDA_PKL, "(exists)" if LDA_PKL.is_file() else "(missing — per-cluster block may be absent in JSON)")

if not MODEL_RESULTS_PATH.is_file():
    raise FileNotFoundError(f"Run Step 6 training first. Expected: {MODEL_RESULTS_PATH}")

with open(MODEL_RESULTS_PATH, encoding="utf-8") as f:
    history = json.load(f)

rows = []
for bandit_key, rounds in history.items():
    for round_idx, rec in enumerate(rounds):
        vpc = rec.get("validation_per_cluster")
        base = {
            "bandit": bandit_key,
            "round": round_idx + 1,
            "eval_f1_pos": rec.get("eval_f1_pos"),
            "eval_accuracy": rec.get("eval_accuracy"),
        }
        if not vpc or not isinstance(vpc, dict):
            rows.append({**base, "lda_cluster": None, "note": "no validation_per_cluster — rebuild LDA with .pkl"})
            continue
        if vpc.get("error"):
            rows.append({**base, "lda_cluster": None, "note": f"error: {vpc.get('error')}"})
            continue
        for cid, stats in vpc.items():
            row = {**base, "lda_cluster": cid}
            if isinstance(stats, dict):
                for k, v in stats.items():
                    row[f"val_{k}"] = v
            rows.append(row)

df = pd.DataFrame(rows)
if df.empty:
    print("No rows to display.")
else:
    sort_cols = [c for c in ["bandit", "round", "lda_cluster"] if c in df.columns]
    display(df.sort_values(sort_cols, na_position="last"))


## Step 6c: Plots (validation metrics)

Loads `{TRAIN_FILE}_binary_{TARGET_LABEL}_model_results.json`, saves figures under **`logs/figures/`** on Drive, and shows them inline (small PNGs to avoid Colab stalls).


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%cd "{EXPERIMENT_ROOT}"

METRIC_KEY = "eval_f1_pos"  # matches default -metric f1_pos in training cell


def _to_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return np.nan


MODEL_RESULTS_PATH = Path(EXPERIMENT_ROOT) / f"{TRAIN_FILE}_binary_{TARGET_LABEL}_model_results.json"
FIG_DIR = Path(EXPERIMENT_ROOT) / "logs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

if not MODEL_RESULTS_PATH.is_file():
    print("No results file yet — run Step 6 training first:", MODEL_RESULTS_PATH)
else:
    with open(MODEL_RESULTS_PATH, encoding="utf-8") as f:
        history = json.load(f)

    rows = []
    for bandit_key, rounds in history.items():
        if not isinstance(rounds, list):
            continue
        for i, rec in enumerate(rounds):
            if not isinstance(rec, dict):
                continue
            rows.append(
                {
                    "bandit": str(bandit_key),
                    "round": i + 1,
                    "eval_f1_pos": _to_float(rec.get("eval_f1_pos")),
                    "eval_accuracy": _to_float(rec.get("eval_accuracy")),
                    "eval_f1_macro": _to_float(rec.get("eval_f1_macro")),
                    "eval_loss": _to_float(rec.get("eval_loss")),
                }
            )

    metrics_df = pd.DataFrame(rows)
    if metrics_df.empty:
        print("JSON loaded but no metric rows parsed.")
    else:
        fig, axes = plt.subplots(2, 2, figsize=(11, 8))

        ax = axes[0, 0]
        for b, g in metrics_df.groupby("bandit"):
            g = g.sort_values("round")
            ax.plot(g["round"], g["eval_f1_pos"], marker="o", label=str(b))
        ax.set_title("Validation F1 (positive)")
        ax.set_xlabel("Round index (within bandit)")
        ax.set_ylabel(METRIC_KEY)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8, loc="best")

        ax = axes[0, 1]
        for b, g in metrics_df.groupby("bandit"):
            g = g.sort_values("round")
            ax.plot(g["round"], g["eval_accuracy"], marker="o", label=str(b))
        ax.set_title("Validation accuracy")
        ax.set_xlabel("Round")
        ax.set_ylabel("eval_accuracy")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8, loc="best")

        ax = axes[1, 0]
        best_per_bandit = metrics_df.groupby("bandit", as_index=False)["eval_f1_pos"].max()
        ax.bar(best_per_bandit["bandit"].astype(str), best_per_bandit["eval_f1_pos"], color="steelblue")
        ax.set_title("Best validation F1(pos) per bandit")
        ax.set_xlabel("Bandit / sampling key")
        ax.set_ylabel("max eval_f1_pos")
        ax.tick_params(axis="x", rotation=45)

        ax = axes[1, 1]
        best_idx = metrics_df["eval_f1_pos"].idxmax()
        bkey = str(metrics_df.loc[best_idx, "bandit"])
        ridx = int(metrics_df.loc[best_idx, "round"]) - 1
        rounds_list = history.get(bkey)
        if rounds_list is None:
            for k in history:
                if str(k) == bkey:
                    rounds_list = history[k]
                    break
        vpc = None
        if isinstance(rounds_list, list) and 0 <= ridx < len(rounds_list):
            rec_pick = rounds_list[ridx]
            if isinstance(rec_pick, dict):
                vpc = rec_pick.get("validation_per_cluster")
        clusters, fps = [], []
        if isinstance(vpc, dict) and not vpc.get("error"):
            for cid, st in sorted(vpc.items(), key=lambda x: str(x[0])):
                if not isinstance(st, dict) or st.get("f1_pos") is None:
                    continue
                clusters.append(str(cid))
                fps.append(_to_float(st["f1_pos"]))
        if not clusters:
            ax.text(
                0.5,
                0.5,
                "No per-cluster F1\n(use *_lda.pkl + multiclass CSVs)",
                ha="center",
                va="center",
            )
            ax.axis("off")
        else:
            ax.bar(clusters, fps, color="seagreen")
            ax.set_title(f'Per-LDA-cluster val F1(pos)\n(best global round: bandit={bkey}, round={ridx + 1})')
            ax.set_xlabel("LDA cluster id")
            ax.set_ylabel("f1_pos")

        plt.tight_layout()
        out_path = FIG_DIR / "active_learning_metrics.png"
        plt.savefig(out_path, dpi=120, bbox_inches="tight")
        print("Saved:", out_path)
        plt.show()


## Step 7: Final Evaluation on Test Set

Use **Step 6b** to compare validation performance **by LDA cluster**; here we score **held-out test** examples.

Select the best checkpoint from training and evaluate on the held-out test set.

After running the optional random baseline, set **`EVAL_RUN_TAG = "random"`** in the Step 7 code cell (same **`TEST_FILE`**) to evaluate that model.

**Metrics reported:**
- Positive-class (love) precision, recall, F1
- Macro-averaged F1
- Accuracy
- Mean confidence score

In [ ]:
%cd "{EXPERIMENT_ROOT}"
!ls -la models

In [ ]:
# Folder must contain config.json (see training logs for best eval_f1_pos & bandit id).
# Same TEST_FILE for every run; only the checkpoint folder changes with EVAL_RUN_TAG.
EVAL_RUN_TAG = ""  # "" = Thompson default; "random" = optional baseline cell above
_ckpt_infix = f"{EVAL_RUN_TAG}_" if EVAL_RUN_TAG else ""
CHECKPOINT_DIR = os.path.join(
    EXPERIMENT_ROOT,
    "models",
    f"binary_{TARGET_LABEL}_{_ckpt_infix}fine_tunned_0_bandit_0",
)

eval_cmd = [
    "python",
    "-u",
    "src/eval_emotion_binary.py",
    "-test_path",
    TEST_FILE,
    "-model_path",
    CHECKPOINT_DIR,
    "-target_emotion",
    TARGET_LABEL,
    "-base_model",
    "bert-base-uncased",
    "-max_length",
    "128",
]
run_logged(
    eval_cmd,
    os.path.join(LOG_DIR, "eval_test.log"),
    cwd=EXPERIMENT_ROOT,
    notebook_tail_lines=None,
)
